Weather is temperal, therfore we do not do random spliting 

In [52]:
import numpy as np
import pandas as pd
import joblib
import os 

from sklearn.ensemble import (
    GradientBoostingRegressor,
    GradientBoostingClassifier,
 #   RandomForestRegressor ( Not needed for deployed models )
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    classification_report,
    roc_auc_score
)
#The following imports are needed when the Random Forest Trees where to big to deploy and a replacement with less resource useage  was required.
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error


In [53]:


# Load data
df = pd.read_csv("Cleaned1_WeatherDataset.csv")

# Keep chronological order
df = df.reset_index(drop=True)

# Time-based split (80% train, 20% test)
split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx]
test_df  = df.iloc[split_idx:]

So  we keep all  the features, and drop the vairble that we are going to be predicting to avoid leakage 

In [54]:
feature_cols = df.columns.tolist()


Temperature Forecasting 

In [55]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

X_train = train_df.drop(columns=["Temp_C"])
y_train = train_df["Temp_C"]

X_test  = test_df.drop(columns=["Temp_C"])
y_test  = test_df["Temp_C"]

temp_model = GradientBoostingRegressor(random_state=42)
temp_model.fit(X_train, y_train)

temp_preds = temp_model.predict(X_test)

mae = mean_absolute_error(y_test, temp_preds)
rmse = np.sqrt(mean_squared_error(y_test, temp_preds))

print("Temperature MAE:", mae)
print("Temperature RMSE:", rmse)


Temperature MAE: 0.0765043203992501
Temperature RMSE: 0.09339771879070816


Visability Forecasting 
So we already had a high_visability which is perfect for our binary target. 
We then also dropped the Visability_Km as this then avoids target leakage 
A benifit as well with this modelis that tree models handle extreme visibility drops naturally

In [56]:

from sklearn.metrics import classification_report, roc_auc_score

X_train = train_df.drop(columns=["high_visibility", "Visibility_km"])
y_train = train_df["high_visibility"]

X_test  = test_df.drop(columns=["high_visibility", "Visibility_km"])
y_test  = test_df["high_visibility"]

vis_model = GradientBoostingClassifier(random_state=42)
vis_model.fit(X_train, y_train)

vis_preds = vis_model.predict(X_test)

print(classification_report(y_test, vis_preds))
print("Visibility ROC‑AUC:",
      roc_auc_score(y_test, vis_model.predict_proba(X_test)[:, 1]))

              precision    recall  f1-score   support

           0       0.94      0.94      0.94      1601
           1       0.38      0.36      0.37       156

    accuracy                           0.89      1757
   macro avg       0.66      0.65      0.66      1757
weighted avg       0.89      0.89      0.89      1757

Visibility ROC‑AUC: 0.8731361809125707


Precipitation forecasting (Rain / Snow)

Rain Occurance 

In [57]:
from sklearn.ensemble import GradientBoostingClassifier

X_train = train_df.drop(columns=["Rain"])
y_train = train_df["Rain"]

X_test  = test_df.drop(columns=["Rain"])
y_test  = test_df["Rain"]

rain_model = GradientBoostingClassifier(random_state=42)
rain_model.fit(X_train, y_train)

rain_preds = rain_model.predict(X_test)
print(classification_report(y_test, rain_preds))

              precision    recall  f1-score   support

           0       0.99      0.98      0.98      1661
           1       0.69      0.88      0.77        96

    accuracy                           0.97      1757
   macro avg       0.84      0.93      0.88      1757
weighted avg       0.98      0.97      0.97      1757



Snow Occurance

In [58]:
X_train = train_df.drop(columns=["Snow"])
y_train = train_df["Snow"]

X_test  = test_df.drop(columns=["Snow"])
y_test  = test_df["Snow"]

snow_model = GradientBoostingClassifier(random_state=42)
snow_model.fit(X_train, y_train)

snow_preds = snow_model.predict(X_test)
print(classification_report(y_test, snow_preds))

              precision    recall  f1-score   support

           0       0.95      0.99      0.97      1588
           1       0.83      0.51      0.63       169

    accuracy                           0.94      1757
   macro avg       0.89      0.75      0.80      1757
weighted avg       0.94      0.94      0.94      1757



Now for the Wind Speed and Pressure, we will be using a different model. We will use the Random Forest Tree regressor model  to  to get more accurate results 

We also use the following process below, as it does a time-aware split, seperate models per weather varible, tree‑based methods for non‑linear, no data leakageatmospheric physics, correct evaluation metrics

In [66]:
# this was our orginal choice, but due to deployment issues, being limited resources, the code that has been run is the suitabel replacement

from sklearn.ensemble import RandomForestRegressor

X_train = train_df.drop(columns=["Wind Speed_km/h"])
y_train = train_df["Wind Speed_km/h"]

X_test  = test_df.drop(columns=["Wind Speed_km/h"])
y_test  = test_df["Wind Speed_km/h"]

wind_model = RandomForestRegressor(n_estimators=200, random_state=42)
wind_model.fit(X_train, y_train)

preds = wind_model.predict(X_test)
print(f"{"Wind Speed_km/h"} MAE:", mean_absolute_error(y_test, preds))

X_train = train_df.drop(columns=["Press_kPa"])
y_train = train_df["Press_kPa"]

X_test  = test_df.drop(columns=["Press_kPa"])
y_test  = test_df["Press_kPa"]

pressure_model = RandomForestRegressor(n_estimators=200, random_state=42)
pressure_model.fit(X_train, y_train)

preds = pressure_model.predict(X_test)
print(f"{"Press_kPa"} MAE:", mean_absolute_error(y_test, preds))

Wind Speed_km/h MAE: 0.7843771117947085
Press_kPa MAE: 0.8448651085509329


In [65]:

os.makedirs("models", exist_ok=True)

joblib.dump(temp_model, "models/temperature_forecast_model.pkl")
joblib.dump(vis_model, "models/high_visibility_classifier.pkl")
joblib.dump(rain_model, "models/rain_classifier.pkl")
joblib.dump(snow_model, "models/snow_classifier.pkl")
joblib.dump(wind_model, "models/wind_speed_regressor.pkl",compress=9) # the compress part, compresses the model so that it is more workable file size wise for deployment 
joblib.dump(pressure_model, "models/pressure_regressor.pkl",compress=9)


['models/pressure_regressor.pkl']